In [1]:
from experiments.bench import harness, plots
import pandas as pd
result = harness.load_result("exp4_rag")
print(result["environment"])

{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'machine': 'arm64', 'cpu_count': 10, 'cpu_model': 'Apple M4', 'ram': '16.0 GiB', 'cbc': 'Version: 2.10.3', 'neo4j_server': 'Neo4j Kernel 5.26.28 (community)', 'packages': {'numpy': '2.5.1', 'scipy': '1.18.0', 'pulp': '3.3.2', 'neo4j': '6.2.0', 'tiktoken': '0.13.0', 'openai': '2.52.0', 'pandas': '3.0.5'}}


In [2]:
df = pd.DataFrame(result["data"]["rows"])
lat = df.pivot_table(index=["query", "n_people"], columns="backend",
                     values="median_ms")
lat["neo4j/sqlite"] = lat["neo4j"] / lat["sqlite"]
lat["결과수"] = df.pivot_table(index=["query", "n_people"],
                             columns="backend", values="result_count")["sqlite"]
display(lat.round(4))

backend                            neo4j  sqlite  neo4j/sqlite     결과수
query                  n_people                                       
overfamiliar_pairs     100        1.8102  0.0946       19.1304   137.0
                       300        4.6983  0.2998       15.6742   434.0
                       500        8.7350  0.5248       16.6434   747.0
                       1000      15.9463  0.9812       16.2524  1390.0
replacement_candidates 100        0.5030  0.0167       30.0690     3.0
                       300        0.4198  0.0449        9.3494     2.0
                       500        0.4719  0.0835        5.6517     4.0
                       1000       0.4571  0.1759        2.5985     6.0
skill_within_hops      100        0.4371  0.0278       15.7045     3.0
                       300        0.4023  0.0340       11.8327     5.0
                       500        0.4968  0.0544        9.1364     6.0
                       1000       0.4170  0.0635        6.5667     5.0
swap_diff              100        0.6531  0.0262       24.9184    25.0
                       300        0.6006  0.0230       26.1389    21.0
                       500        0.5741  0.0231       24.8244    20.0
                       1000       0.6043  0.0239       25.3125    22.0
team_cohesion          100        0.6539  0.0277       23.6005    10.0
                       300        0.4914  0.0201       24.4173     4.0
                       500        0.6509  0.0234       27.7733     4.0
                       1000       0.5078  0.0177       28.6767     2.0

In [3]:
plots.exp4_rag(result)   # figures/exp4_rag.png 생성

PosixPath('/Users/honey/Dev/teamweaver/experiments/figures/exp4_rag.png')

In [4]:
from experiments import decision
from experiments import report as _report
from IPython.display import Markdown, display

# 마크다운 셀에 수치를 손으로 적지 않는다 — report.py의 헬퍼를 그대로 재사용해
# 리포트와 노트북이 같은 소스에서 같은 로직으로 같은 문장을 만든다.
d = result["data"]
rows = d["rows"]
wins, cells = _report._exp4_tally(rows)
sep = _report._exp4_separation(rows)
cal = d["calibration"]
flip_sess, _ = _report._exp4_flip_count(rows, cal["neo4j_session_open_ms"])
flip_floor, _ = _report._exp4_flip_count(
    rows, cal["neo4j_session_plus_trivial_query_ms"])
growing, flat = _report._exp4_output_growth(rows)
bad_parity = [p for p in d["parity"] if not p["match"]]

lines = ["## 관찰 요약", ""]
lines.append(f"- **의사결정 규칙 1이 소비하는 수치: {cells}개 셀 중 Neo4j 우위 "
             f"{wins}개** — 사전 고정 임계 {decision.MAJORITY_CELLS} 미달.")
if sep:
    clean, total, name, ratio = sep
    lines.append(f"- 표본 겹침 없는 셀 {clean}/{total} — 가장 근접한 셀조차 "
                 f"`{name}`에서 {ratio:.1f}배 차이라 추정량을 바꿔도 뒤집히지 않는다.")
lines.append(f"- 호출당 고정비를 Neo4j 수치에서 빼도 뒤집히는 셀은 세션 획득분 "
             f"{flip_sess}/{cells}, 고정 바닥 전체(과보정) {flip_floor}/{cells}다.")
lines.append(f"- 출력이 규모를 따라 자라는 질의는 {_report._fmt_growth(growing)} 뿐이고 "
             f"나머지({_report._fmt_growth(flat)})는 사실상 고정 출력이다 — "
             f"n={max(r['n_people'] for r in rows)}을 넘어 외삽하지 말 것.")
lines.append(f"- `data.parity`는 "
             f"{'전 조합 일치' if not bad_parity else f'{len(bad_parity)}건 불일치'}"
             "이므로 위 지연 차이는 결과 집합 크기 차이가 아니다. "
             f"`data.skipped`는 {'비어 있다' if not d['skipped'] else d['skipped']}.")

display(Markdown("\n".join(lines)))

## 관찰 요약

- **의사결정 규칙 1이 소비하는 수치: 20개 셀 중 Neo4j 우위 0개** — 사전 고정 임계 11 미달.
- 표본 겹침 없는 셀 20/20 — 가장 근접한 셀조차 `replacement_candidates @ n=1000`에서 2.4배 차이라 추정량을 바꿔도 뒤집히지 않는다.
- 호출당 고정비를 Neo4j 수치에서 빼도 뒤집히는 셀은 세션 획득분 0/20, 고정 바닥 전체(과보정) 1/20다.
- 출력이 규모를 따라 자라는 질의는 `overfamiliar_pairs`(137→1390행) 뿐이고 나머지(`replacement_candidates`(3→6행), `skill_within_hops`(3→5행), `swap_diff`(25→22행), `team_cohesion`(10→2행))는 사실상 고정 출력이다 — n=1000을 넘어 외삽하지 말 것.
- `data.parity`는 전 조합 일치이므로 위 지연 차이는 결과 집합 크기 차이가 아니다. `data.skipped`는 비어 있다.